In [1]:
!pip install -q \
    transformers==4.53.3 \
    peft==0.16.0 \
    trl==0.20.0 \
    accelerate==1.10.1
import os
import gc
import torch
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# Config
# ============================================================

MODEL_NAME = "Qwen/Qwen3-0.6B"
# OUTPUT_DIR = "./qwen3_0_6b_event_extractor"
OUTPUT_DIR = "/content/drive/MyDrive/google-colab/training_lstm/qwen3_0_6b_event_extractor"
# DATASET_PATH = "./synthetic_event_extraction_dataset_2000.json"
DATASET_PATH = "/content/drive/MyDrive/google-colab/training_lstm/train.json"

# ============================================================
# Cleanup Memory
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token

# ============================================================
# Load Base Model
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto"
)

model.gradient_checkpointing_enable()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.config.pad_token_id = tokenizer.pad_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

# ============================================================
# LoRA Setup
# ============================================================

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# ============================================================
# Load Dataset
# ============================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# ============================================================
# Format Dataset
# ============================================================

def format_example(item):
    return {
        "text": (
            "Task: Extract event details as raw JSON.\n"
            f"Input: \"{item.get('input', '')}\"\n"
            f"Output: {json.dumps(item.get('output', {}), ensure_ascii=False)}"
            f"{tokenizer.eos_token}"
        )
    }

dataset = Dataset.from_list(
    [format_example(x) for x in raw_data]
)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")

trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756
Train examples: 4000
Eval examples: 1000


In [5]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch",
    report_to="none",
    fp16=False,
    bf16=False,
    max_length=256,
    completion_only_loss=True,
    dataset_text_field="text",
    eval_strategy="epoch",
    packing=False,
)

print("🚀 Starting training...")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)

# ============================================================
# Sanity Check
# ============================================================

batch = next(iter(trainer.get_train_dataloader()))

print("Batch Keys:", batch.keys())
print("Input Shape:", batch["input_ids"].shape)
print("Attention Shape:", batch["attention_mask"].shape)
print("Labels Shape:", batch["labels"].shape)

# ============================================================
# Train
# ============================================================

print("🚀 Starting LoRA training...")

trainer.train()

# ============================================================
# Save LoRA Adapter
# ============================================================

print("💾 Saving LoRA adapter...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ LoRA adapter saved!")

# ============================================================
# Optional: Merge LoRA into Full Model
# ============================================================

merged_model = model.merge_and_unload()

MERGED_DIR = OUTPUT_DIR + "_merged"

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("✅ Merged model saved!")
print(MERGED_DIR)


🚀 Starting training...


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Batch Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input Shape: torch.Size([2, 146])
Attention Shape: torch.Size([2, 146])
Labels Shape: torch.Size([2, 146])
🚀 Starting LoRA training...


Epoch,Training Loss,Validation Loss
1,0.066600,0.064557
2,0.060400,0.062906
3,0.066300,0.062568
4,0.061700,0.061973
5,0.061200,0.061849


💾 Saving LoRA adapter...
✅ LoRA adapter saved!
✅ Merged model saved!
/content/drive/MyDrive/google-colab/training_lstm/qwen3_0_6b_event_extractor_merged


In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_loss = []
eval_loss = []
steps = []
eval_steps = []

for log in logs:
    if "loss" in log and "eval_loss" not in log:
        train_loss.append(log["loss"])
        steps.append(log["step"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])
        eval_steps.append(log.get("step", len(eval_loss)))

# Plot training loss
plt.figure()
plt.plot(steps, train_loss, label="Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.show()

# Plot eval loss
plt.figure()
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Step / Epoch")
plt.ylabel("Loss")
plt.title("Validation Loss Curve")
plt.legend()
plt.show()